# 🔁 Milestone 4 — CRNN: CNN + LSTM/GRU
**Sequential model that captures both local patterns AND temporal structure**

### Why CRNN?
- CNN alone treats each time frame independently
- Music has **temporal structure** (verses, chorus, rhythm over time)
- LSTM reads the CNN's output as a sequence → captures musical flow
- CRNN = CNN extracts features per time frame → LSTM reads the sequence

### Architecture:
```
Input (1, 128, 256)
    ↓ CNN blocks → (256, 1, T)
    ↓ Reshape → sequence of T vectors of size 256
    ↓ LSTM/GRU → hidden states
    ↓ Take last hidden state
    ↓ Linear → 10 class scores
```

In [ ]:
!pip install librosa wandb -q

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')

BASE        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUPS_DIR = f'{BASE}/mashups'
TEST_CSV    = f'{BASE}/test.csv'
GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
ROLL_NO     = 'YOUR_ROLL_NO'  # ⚠️ change this!
USE_STEMS   = ['vocals', 'drums', 'bass']

## 1️⃣ Audio Processing (same as Milestone 3)

In [ ]:
CONFIG = {
    'sample_rate'  : 22050,
    'duration'     : 30,
    'n_mels'       : 128,
    'n_fft'        : 2048,
    'hop_length'   : 512,
    'target_length': 256,
    'batch_size'   : 32,
    'epochs'       : 35,
    'lr'           : 5e-4,
    'dropout'      : 0.3,
    'rnn_hidden'   : 256,
    'rnn_layers'   : 2,
    'rnn_type'     : 'GRU',   # 'GRU' or 'LSTM'
    'model'        : 'CRNN',
}

def audio_to_melspec(path, cfg=CONFIG):
    try:
        y, sr = librosa.load(path, sr=cfg['sample_rate'], duration=cfg['duration'])
        if len(y) < sr: y = np.zeros(sr * cfg['duration'])
        mel    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=cfg['n_mels'],
                    n_fft=cfg['n_fft'], hop_length=cfg['hop_length'])
        mel_db = librosa.power_to_db(mel, ref=np.max)
        T = cfg['target_length']
        if mel_db.shape[1] < T:
            mel_db = np.pad(mel_db, ((0,0),(0, T - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :T]
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db[np.newaxis, :, :]
    except:
        return np.zeros((1, CONFIG['n_mels'], CONFIG['target_length']))

class AudioDataset(Dataset):
    def __init__(self, files, labels, augment=False):
        self.files = files; self.labels = labels; self.augment = augment
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        mel = audio_to_melspec(self.files[idx])
        if self.augment:
            t = random.randint(0, CONFIG['target_length']-25)
            mel[0, :, t:t+20] = 0
            f = random.randint(0, CONFIG['n_mels']-12)
            mel[0, f:f+10, :] = 0
        return torch.FloatTensor(mel), torch.tensor(self.labels[idx], dtype=torch.long)

class TestDataset(Dataset):
    def __init__(self, files): self.files = files
    def __len__(self): return len(self.files)
    def __getitem__(self, idx): return torch.FloatTensor(audio_to_melspec(self.files[idx]))

# Build file lists
all_files, all_labels = [], []
for genre in GENRES:
    for song in sorted(os.listdir(f'{STEMS_DIR}/{genre}')):
        for stem in USE_STEMS:
            path = f'{STEMS_DIR}/{genre}/{song}/{stem}.wav'
            if os.path.exists(path):
                all_files.append(path); all_labels.append(genre)

le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)
X_tr, X_val, y_tr, y_val = train_test_split(
    all_files, all_labels_enc, test_size=0.2, stratify=all_labels_enc, random_state=42)

train_loader = DataLoader(AudioDataset(X_tr, y_tr, augment=True),
                          batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader   = DataLoader(AudioDataset(X_val, y_val),
                          batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
print(f'Train: {len(X_tr)}, Val: {len(X_val)}')